In [9]:
import jax.numpy as jnp
import jax
import jax.random as jrandom

from diffrax.STLA.tree import VirtualSTLATree
from jax import config
config.update("jax_debug_nans", True)

In [33]:
key = jrandom.PRNGKey(200)
shape_struct = jax.ShapeDtypeStruct((10000,), jnp.float32)
tree1 = VirtualSTLATree(t0=0, t1=1, tol=2**-7, shape=shape_struct, key=key)

In [34]:
def print_correct_vars(_t0, _t1):
    h = _t1 - _t0
    print(f"true hh_var: {1/12 * h:.4f}, w_var: {h:.4f}")

In [36]:
t0, t1, t2, t3 = 0.1256, 0.4061, 0.56264, 0.7533331
w02, hh02 = tree1.evaluate(t0,t2)
w13, hh13 = tree1.evaluate(t1,t3)
w01, hh01 = tree1.evaluate(t0,t1)
w23, hh23 = tree1.evaluate(t2,t3)
w03, hh03 = tree1.evaluate(t0,t3)
wtot, hhtot = tree1.evaluate(0,1)

In [37]:
for w, hh in [(w02, hh02), (w13, hh13), (w01,hh01), (w23, hh23), (wtot,hhtot)]:
    print(f"hh_mean: {jnp.mean(hh):.6f}, w_mean: {jnp.mean(w):.6f}, hh_var: {jnp.var(hh):.5f}, w_var: {jnp.var(w):.5f}")

hh_mean: 0.000143, w_mean: -0.004483, hh_var: 0.03621, w_var: 0.45068
hh_mean: -0.000281, w_mean: -0.006740, hh_var: 0.02895, w_var: 0.34776
hh_mean: 0.000233, w_mean: -0.002754, hh_var: 0.02363, w_var: 0.28614
hh_mean: -0.001544, w_mean: -0.005010, hh_var: 0.01580, w_var: 0.19272
hh_mean: -0.001843, w_mean: -0.011876, hh_var: 0.08470, w_var: 1.00171


In [38]:
print_correct_vars(t0, t2)
print(f"emp  hh_var: {jnp.var(hh02):.4f}, w_var: {jnp.var(w02):.4f}")
print_correct_vars(t1, t3)
print(f"emp  hh_var: {jnp.var(hh13):.4f}, w_var: {jnp.var(w13):.4f}")
print_correct_vars(t0, t1)
print(f"emp  hh_var: {jnp.var(hh01):.4f}, w_var: {jnp.var(w01):.4f}")
print_correct_vars(t2, t3)
print(f"emp  hh_var: {jnp.var(hh23):.4f}, w_var: {jnp.var(w23):.4f}")

true hh_var: 0.0364, w_var: 0.4370
emp  hh_var: 0.0362, w_var: 0.4507
true hh_var: 0.0289, w_var: 0.3472
emp  hh_var: 0.0290, w_var: 0.3478
true hh_var: 0.0234, w_var: 0.2805
emp  hh_var: 0.0236, w_var: 0.2861
true hh_var: 0.0159, w_var: 0.1907
emp  hh_var: 0.0158, w_var: 0.1927


In [41]:
print(jnp.cov(w02, w13))
print(jnp.cov(hh02, hh13))

[[0.45072773 0.16195504]
 [0.16195504 0.3477899 ]]
[[ 0.0362108  -0.01186689]
 [-0.01186689  0.02895591]]


In [23]:
# Chen's relation
w_diff_013 = jnp.mean(jnp.abs(w03 - (w01 + w13)))
hh_chen_01_03 = (t1-t0)*hh01 + (t3-t1)*hh13 + 0.5*((t3-t0)*w01 - (t1-t0) * (w01 + w13))
hh_diff_013 = jnp.mean(jnp.abs((t3 - t0) * hh03 - hh_chen_01_03))
w_diff_023 = jnp.mean(jnp.abs(w03 - (w02 + w23)))
print(w_diff_013)
print(hh_diff_013)
print(w_diff_023)

1.1658482e-08
1.5718944e-08
1.0407902e-08
